# Day 200 — Month 13, Week 1, Day 1: LangGraph Fundamentals
### Stateful Graphs, Conditional Edges, Checkpointing, Human-in-the-Loop

**Program:** Agentic AI & Advanced GenAI Portfolio (Month 13)
**Builds on:** Month 10 (LangChain + Groq stack)
**Feeds into:** Month 13 Week 4 capstone — a deployable support chatbot (3rd standalone starred GitHub repo)

**Scenario:** You're building the routing brain for a support-ticket triage agent.
Some tickets an agent can resolve on its own. Some need a human to look at them
before anything gets sent. LangGraph is the tool for exactly this: a workflow
that branches, pauses for a human, and remembers where it left off.

---
**Version note (checked live, July 2026):** Groq deprecated `llama-3.1-8b-instant`
on June 17, 2026. This notebook uses `openai/gpt-oss-20b`, Groq's recommended
replacement — free tier, similar latency. `langgraph` is pinned fresh at 1.2.8
(LangGraph hit a stable 1.0 release since Month 10). We build directly on
`StateGraph` / `MemorySaver` and skip `langgraph.prebuilt` — that module had
breaking changes between recent patch versions, and for *fundamentals* the
low-level primitives are what you actually need to understand anyway.


In [1]:
# ── SETUP ──────────────────────────────────────────────────────────────
# Run this cell, then Runtime → Restart session, then run everything from the top.
!pip install -q langgraph==1.2.8 langchain-groq==1.1.2 --break-system-packages 2>/dev/null || \
!pip install -q langgraph==1.2.8 langchain-groq==1.1.2

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Setup cell complete. If this is your first run, RESTART the runtime now, then run all cells from the top.")


Setup cell complete. If this is your first run, RESTART the runtime now, then run all cells from the top.


---
## SECTION 1 — Raw Data (locked — never modify)

`SUPPORT_TICKETS` — TeleServe-style support queue, seed=200, 12 tickets.
This is your immutable source data for today. Every task below reads from
this list. Do not edit the values inside it.


In [2]:
# ── RAW DATA — LOCKED. DO NOT MODIFY. ────────────────────────────────────
import random
random.seed(200)

SUPPORT_TICKETS = [
    {"ticket_id": "T-2001", "customer": "Rohit Malhotra", "category": "billing",
     "text": "I was charged twice for my monthly plan this cycle. Please refund the duplicate charge.",
     "urgency": "medium"},
    {"ticket_id": "T-2002", "customer": "Ayesha Khan", "category": "technical",
     "text": "My internet has been completely down for 6 hours. This is urgent, I work from home.",
     "urgency": "high"},
    {"ticket_id": "T-2003", "customer": "Suresh Iyer", "category": "billing",
     "text": "What is the due date for my next invoice?",
     "urgency": "low"},
    {"ticket_id": "T-2004", "customer": "Neha Bansal", "category": "account",
     "text": "I want to permanently close my account and get written confirmation of cancellation.",
     "urgency": "high"},
    {"ticket_id": "T-2005", "customer": "Karan Mehta", "category": "technical",
     "text": "How do I reset my WiFi router to factory settings?",
     "urgency": "low"},
    {"ticket_id": "T-2006", "customer": "Divya Nair", "category": "billing",
     "text": "Can you explain the taxes line item on my bill? The amount looks higher than usual.",
     "urgency": "medium"},
    {"ticket_id": "T-2007", "customer": "Arjun Reddy", "category": "technical",
     "text": "Internet keeps disconnecting every 10 minutes since yesterday, very frustrating.",
     "urgency": "high"},
    {"ticket_id": "T-2008", "customer": "Priyanka Joshi", "category": "account",
     "text": "I'd like to upgrade my plan to the 200 Mbps tier, what's the price difference?",
     "urgency": "low"},
    {"ticket_id": "T-2009", "customer": "Vikram Chauhan", "category": "billing",
     "text": "I am extremely unhappy — this is the third billing error in three months. I want a manager.",
     "urgency": "high"},
    {"ticket_id": "T-2010", "customer": "Meera Pillai", "category": "technical",
     "text": "Can you confirm the standard installation timeline for a new connection?",
     "urgency": "low"},
    {"ticket_id": "T-2011", "customer": "Sanjay Rao", "category": "account",
     "text": "Someone else's name is showing on my account profile page, please investigate.",
     "urgency": "high"},
    {"ticket_id": "T-2012", "customer": "Ritu Kapoor", "category": "billing",
     "text": "Just confirming — my autopay went through fine this month, no action needed, thanks.",
     "urgency": "low"},
]

print(f"Loaded {len(SUPPORT_TICKETS)} tickets (locked).")
print(f"Categories: {set(t['category'] for t in SUPPORT_TICKETS)}")
print(f"Urgency levels: {set(t['urgency'] for t in SUPPORT_TICKETS)}")


Loaded 12 tickets (locked).
Categories: {'billing', 'technical', 'account'}
Urgency levels: {'low', 'high', 'medium'}


---
## SECTION 2 — Concept Notes

### What LangGraph actually is
A **plain LangChain chain runs in one direction, once.** A **LangGraph graph**
is a state machine — it can loop, branch, pause, and resume. That's the entire
reason it exists: real agent workflows aren't straight lines, they're
decision trees with waiting points.

### The five pieces you need today

**1. State schema** — a `TypedDict` describing what data flows through the
graph. Every node reads from it and returns updates to merge into it.
```python
from typing import TypedDict

class TicketState(TypedDict):
    ticket_id: str
    text: str
    category: str
    urgency: str
    classification: str   # filled in by a node
    response: str          # filled in by a node
    status: str             # filled in by a node
```

**2. Nodes** — plain Python functions. Each takes the state dict, does
something (call an LLM, run logic), and returns a dict of the fields it wants
to update.
```python
def classify_node(state: TicketState) -> dict:
    # ... call the LLM, decide something ...
    return {"classification": "auto_resolvable"}
```

**3. Conditional edges** — a routing function that inspects the state and
returns the *name* of the next node. This is what makes it a graph and not a
chain — the path taken depends on the data.
```python
def route_ticket(state: TicketState) -> str:
    if state["urgency"] == "high":
        return "escalate"
    return "auto_respond"
```

**4. Checkpointing** — a `MemorySaver` (or a database-backed saver in
production) that snapshots state after every node. Pass a `thread_id` in the
config and the graph remembers exactly where a given conversation/ticket left
off — even across separate `.invoke()` calls. This is what "durable state"
means in the Month 13 memory notes.

**5. Human-in-the-loop** — call `interrupt(payload)` inside a node. Execution
pauses there and returns control to you. You inspect the payload, then resume
by invoking the graph again with a `Command(resume=your_decision)`. The graph
picks up exactly where it paused — nothing upstream re-runs.

### Real-world use
This exact pattern — classify → route → (auto-handle | pause for a human) —
is the backbone of every production support bot, approval workflow, and
content-moderation pipeline. The "auto-resolve low-risk, escalate high-risk"
split is the single most common agentic design pattern in industry right now.

### Why not just use `if/else` in a normal Python script?
You could, for one ticket. The moment you need to **pause mid-workflow and
come back later** (a human takes 20 minutes to review), a normal script has
nowhere to put that state. LangGraph's checkpointer is the piece that makes
"pause now, resume in an hour, from a different process" actually work.


---
## SECTION 3 — Practice Guide

Five tasks. Work top to bottom — each one builds on the last. Fill in every
`# TODO`. Do not look at the Answer Key section until you've attempted each
task.


In [3]:
# ── TASK 1 (15 pts): Define the state schema ─────────────────────────────
# Goal: Define a TypedDict that represents the state flowing through the graph.
# Method: Use typing.TypedDict to specify all fields as strings.

from typing import TypedDict

class TicketState(TypedDict):
    ticket_id: str
    text: str
    category: str
    urgency: str
    classification: str   # filled later: "auto_resolvable" or "needs_human"
    response: str          # draft reply or human decision
    status: str            # "auto_resolved", "human_reviewed", or empty

In [4]:
# ── TASK 2 (25 pts): Build the classify_node and auto_respond_node ──────
# Goal: Use Groq LLM to classify a ticket and generate an auto-reply.
# Method:
#   - classify_node: prompt the LLM to decide if the ticket can be auto-resolved
#     or needs human review, based on urgency and tone.
#   - auto_respond_node: prompt the LLM to draft a short professional reply.

from langchain_groq import ChatGroq

# Instantiate the LLM (Groq recommended replacement)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

def classify_node(state: TicketState) -> dict:
    """Classify ticket as auto_resolvable or needs_human."""
    # Build a prompt that includes the ticket text, category, and urgency
    prompt = (
        f"Ticket text: {state['text']}\n"
        f"Category: {state['category']}\n"
        f"Urgency: {state['urgency']}\n"
        "Based on the above, decide if this ticket can be resolved automatically "
        "or needs human intervention. "
        "Rules: high urgency or angry/complaint tone -> needs_human. "
        "Simple factual questions -> auto_resolvable.\n"
        "Answer exactly one of: 'auto_resolvable' or 'needs_human'."
    )
    response = llm.invoke(prompt)
    classification = response.content.strip().lower()
    # Clean to ensure exact match
    if "auto_resolvable" in classification:
        classification = "auto_resolvable"
    else:
        classification = "needs_human"
    return {"classification": classification}

def auto_respond_node(state: TicketState) -> dict:
    """Generate a short professional reply to the ticket."""
    prompt = (
        f"Ticket text: {state['text']}\n"
        "Draft a short, professional reply to the customer addressing their issue."
    )
    response = llm.invoke(prompt)
    reply = response.content.strip()
    return {"response": reply, "status": "auto_resolved"}

In [5]:
# ── TASK 3 (20 pts): Build the routing function + escalate_node ─────────
# Goal: Route based on classification, and implement human‑in‑the‑loop.
# Method:
#   - route_after_classify: return the next node name based on classification.
#   - escalate_node: call interrupt() with a payload; on resume, store the
#     human decision and update status.

from langgraph.types import interrupt

def route_after_classify(state: TicketState) -> str:
    """Return the name of the next node based on classification."""
    if state["classification"] == "auto_resolvable":
        return "auto_respond"
    else:
        return "escalate"

def escalate_node(state: TicketState) -> dict:
    """Pause for human review and capture the decision."""
    # Prepare the payload for the human reviewer
    payload = {
        "ticket_id": state["ticket_id"],
        "text": state["text"],
        "category": state["category"],
        "urgency": state["urgency"],
    }
    # Interrupt and wait for human input
    human_decision = interrupt(payload)  # This returns the value passed to Command(resume=...)
    return {"response": human_decision, "status": "human_reviewed"}

In [6]:
# ── TASK 4 (20 pts): Wire the graph together ─────────────────────────────
# Goal: Build the StateGraph, add nodes and edges, compile with a checkpointer.
# Method:
#   - Import StateGraph, START, END from langgraph.graph.
#   - Import MemorySaver from langgraph.checkpoint.memory.
#   - Add all nodes, set START→classify, conditional edge from classify,
#     and edges from auto_respond/escalate to END.
#   - Compile with MemorySaver.

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Build the graph
graph = StateGraph(TicketState)
graph.add_node("classify", classify_node)
graph.add_node("auto_respond", auto_respond_node)
graph.add_node("escalate", escalate_node)

# Edges
graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    route_after_classify,
    {
        "auto_respond": "auto_respond",
        "escalate": "escalate"
    }
)
graph.add_edge("auto_respond", END)
graph.add_edge("escalate", END)

# Checkpointer
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("Graph compiled successfully with MemorySaver.")

Graph compiled successfully with MemorySaver.


In [8]:
# ── TASK 5 (20 pts) — CORRECTED ──────────────────────────────────────────
from langgraph.types import Command

# Pick tickets (same as before)
low_ticket = next(t for t in SUPPORT_TICKETS if t["urgency"] == "low")
high_ticket = next(t for t in SUPPORT_TICKETS if t["urgency"] == "high")

print("Low-urgency ticket:", low_ticket["ticket_id"])
print("High-urgency ticket:", high_ticket["ticket_id"])

# ---- Auto-resolve path ----
low_initial = {
    "ticket_id": low_ticket["ticket_id"],
    "text": low_ticket["text"],
    "category": low_ticket["category"],
    "urgency": low_ticket["urgency"],
    "classification": "",
    "response": "",
    "status": ""
}
low_config = {"configurable": {"thread_id": f"ticket-{low_ticket['ticket_id']}"}}

final_low = app.invoke(low_initial, config=low_config)
print("\n--- Auto-resolve result ---")
print(f"Status: {final_low['status']}")
print(f"Response: {final_low['response']}\n")

# ---- Human-in-the-loop path ----
high_initial = {
    "ticket_id": high_ticket["ticket_id"],
    "text": high_ticket["text"],
    "category": high_ticket["category"],
    "urgency": high_ticket["urgency"],
    "classification": "",
    "response": "",
    "status": ""
}
high_config = {"configurable": {"thread_id": f"ticket-{high_ticket['ticket_id']}"}}

# Invoke – returns normally with an __interrupt__ key if paused
result = app.invoke(high_initial, config=high_config)

# Prove the pause directly
snapshot = app.get_state(high_config)
paused = bool(snapshot.next)   # next contains the node(s) waiting to run
interrupt_payload = result.get("__interrupt__")

print("--- Human-in-loop pause verification ---")
print(f"Paused: {paused}")                # Should be True
print(f"Interrupt payload: {interrupt_payload}")  # Shows the dict we passed to interrupt()

# Resume with human decision
human_decision = "Escalating to Tier 2 — refund approved and account review initiated."
final_high = app.invoke(Command(resume=human_decision), config=high_config)

print("\n--- Human-in-loop result ---")
print(f"Status: {final_high['status']}")
print(f"Response: {final_high['response']}")

Low-urgency ticket: T-2003
High-urgency ticket: T-2002

--- Auto-resolve result ---
Status: auto_resolved
Response: Hi [Customer Name],

Your next invoice will be due on **[Due Date]**. If you need any additional details or have further questions, feel free to let us know.

Best regards,  
[Your Name]  
[Company Name] Support Team

--- Human-in-loop pause verification ---
Paused: True
Interrupt payload: [Interrupt(value={'ticket_id': 'T-2002', 'text': 'My internet has been completely down for 6 hours. This is urgent, I work from home.', 'category': 'technical', 'urgency': 'high'}, id='f06a3ef7891446be1d11317af3d8b13e')]

--- Human-in-loop result ---
Status: human_reviewed
Response: Escalating to Tier 2 — refund approved and account review initiated.


---
## SECTION 4 — Scoring Rubric (100 pts)

| Task | What's graded | Points |
|---|---|---|
| Task 1 — State schema | Correct `TypedDict`, all 7 fields, correct types | 15 |
| Task 2 — classify_node + auto_respond_node | LLM called correctly; classify returns exactly one of the two allowed strings; auto_respond returns response + status | 25 |
| Task 3 — Routing + escalate_node | Router returns correct literal node names; `interrupt()` called with a useful payload; correct field updates on resume | 20 |
| Task 4 — Graph wiring | All nodes/edges/conditional edges correctly wired; `MemorySaver` checkpointer attached at compile time | 20 |
| Task 5 — Execution proof | Both paths actually run end-to-end from **printed output**; interrupt genuinely pauses (checked via `get_state`); resume produces the exact resumed value in `response` | 20 |

**NRA reminder:** when you write up what happened in Task 5, don't just
narrate the printed output — apply the framework. *Number* (from the printed
`status` field) → *Reason* (why the routing function sent it there — the
mechanism, stop there) → *Action* (what you'd build next, specific and owned).


---
## Interview Angle

**"Walk me through how you'd design a support ticket triage system with an LLM."**

> I'd model it as a state graph rather than a linear chain, because triage is
> inherently branching — some tickets a model can safely close on its own,
> others need a human before anything goes out. I'd have a classification
> node score risk/urgency, a conditional edge route on that classification,
> and for anything routed to a human reviewer I'd use an interrupt so the
> workflow pauses mid-execution rather than requiring a separate queue system.
> The key production detail is the checkpointer — it persists state per
> thread, so a ticket paused for review can be resumed hours later, by a
> different process, without re-running anything upstream. That durability is
> the actual reason to reach for LangGraph instead of a plain conditional
> script.

---
### What's next
**Day 201 (Week 1, Day 2):** more LangGraph — multi-turn state, looping
edges, and combining checkpointing with a real conversation history (not just
a single-shot ticket).
